# 2. Coordinates: CRS, projections, units, heights: solutions

Exercises of [chapter 2](../../02_coordinates.md). Run the setup cell first.

These are the solutions; try the exercise notebook first.

In [ ]:
# Setup: run this cell first, in every notebook
import os, subprocess, sys
from pathlib import Path

# find the repository: from this folder upwards, the first one holding data_acquisition/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_acquisition").is_dir())
sys.path.insert(0, str(ROOT))                  # so that Python can import the pipeline
SAMPLE = ROOT / "output/manual/sample_root"    # the course's invented dataset
if not SAMPLE.exists():                        # generated once, with this same Python
    subprocess.run([sys.executable, str(ROOT / "docs/manual/tools/make_sample.py"), "--out", str(SAMPLE)], check=True)
DATASET = SAMPLE / "repo/doi_10.5555_training.pond"    # the sample's dataset folder
EXTRACT = ROOT / "docs/manual/data/montana_extract"    # a real piece of Rumney Ranch
os.environ["BEAVER_SITES"] = str(SAMPLE / "sites.yaml")    # use the sample's site registry, not the real one
import warnings; warnings.simplefilter("ignore")           # hide warnings; remove this line when debugging
print("repository:", ROOT); print("sample data root:", SAMPLE)

In [ ]:
import numpy as np
import pyogrio
import rasterio
from pyproj import CRS, Transformer
from rasterio.warp import transform

**1. Read a CRS.** Open `ortho_2023-07-12.tif` of the sample dataset with rasterio. Print `crs`, `crs.to_epsg()` and `crs.to_wkt()`. Is it geographic or projected? What are the units of `res`?

> **From the chapter.** §3: a file can write its CRS as an EPSG code, a PROJ string or a WKT. §4: the CRS fixes the units of the pixel size.
>
> **Code.** No pipeline code: `rasterio`. In the pipeline, `geotiff.info` in [`data_acquisition/readers/geotiff.py`](../../../../data_acquisition/readers/geotiff.py) reads the same values.

In [ ]:
with rasterio.open(DATASET / "ortho_2023-07-12.tif") as r:
    print(r.crs, r.crs.to_epsg())
    print(r.crs.to_wkt()[:200], "...")
    crs = CRS.from_wkt(r.crs.to_wkt())
    print("projected:", crs.is_projected, "| units:", crs.axis_info[0].unit_name, "| res:", r.res)
# A projected CRS (UTM zone 12N) in metres: each pixel is 5 cm by 5 cm.

**2. Degrees and metres.** Transform the top corners of the sample orthomosaic to EPSG:4326 with `pyproj.Transformer`. How wide is the image in degrees of longitude, and in metres? Explain the ratio.

> **From the chapter.** §1: degrees of longitude shrink with latitude. §2: UTM coordinates are metres.
>
> **Code.** No pipeline code: `pyproj.Transformer`. With `always_xy=True` it takes and returns (x, y), so (longitude, latitude).

In [ ]:
with rasterio.open(DATASET / "ortho_2023-07-12.tif") as r:
    b = r.bounds
to_lonlat = Transformer.from_crs("EPSG:32612", "EPSG:4326", always_xy=True)
(lon1, lat1), (lon2, lat2) = to_lonlat.transform(b.left, b.top), to_lonlat.transform(b.right, b.top)
width_m, width_deg = b.right - b.left, lon2 - lon1
print(f"{width_m:.1f} m = {width_deg:.7f} degrees of longitude -> {width_m / width_deg:,.0f} m per degree")
print(f"at the equator one degree is about 111,320 m; here (latitude {lat1:.2f}) about 111,320 x cos(latitude) = "
      f"{111320 * np.cos(np.radians(lat1)):,.0f} m")
# The small difference: the formula assumes a sphere; on the WGS 84 ellipsoid a degree of longitude at this latitude
# is slightly longer.

**3. No CRS.** Open `dem_2022.asc`. What does rasterio report as its CRS? Where in the sample's `metadata.xml` would you find the CRS to put in the summary?

> **From the chapter.** §3: some files have no CRS at all, and it must then come from the dataset's documentation.
>
> **Code.** No pipeline code: `rasterio`, and the text of `DATASET / "metadata.xml"`, the sample's metadata record ([chapter 4](../../04_catalogs_and_metadata.md), §3).

In [ ]:
with rasterio.open(DATASET / "dem_2022.asc") as r:
    print("CRS:", r.crs, "| bounds:", r.bounds)
text = (DATASET / "metadata.xml").read_text()
start = text.find("<methodStep>")
print(text[start:text.find("</methodStep>")])
# No CRS in the file. The record's methods text says "in UTM zone 12N (EPSG:32612)": that goes in the summary
# as `crs: EPSG:32612`, and the item is flagged crs_from_summary.

**4. Heights in feet.** Open `2018-07-26/annotations/dams.gpkg` of the Montana extract with `pyogrio.read_info`. Split its CRS into horizontal and vertical parts (`pyproj.CRS.from_user_input(...).sub_crs_list`). What is the vertical datum, and its unit? How many metres is 5 US survey feet?

> **From the chapter.** §5: a compound CRS is a horizontal CRS plus a vertical one, and the Rumney heights are in US survey feet.
>
> **Code.** No pipeline code: `pyogrio.read_info`, and `pyproj.CRS.from_user_input(...).sub_crs_list` to split the CRS. In the pipeline, `vector._split` in [`data_acquisition/readers/vector.py`](../../../../data_acquisition/readers/vector.py) does this.

In [ ]:
info = pyogrio.read_info(EXTRACT / "2018-07-26/annotations/dams.gpkg")
horizontal, vertical = CRS.from_user_input(info["crs"]).sub_crs_list
print("horizontal:", horizontal.name)
print("vertical:", vertical.name, "| datum:", vertical.datum.name, "| unit:", vertical.axis_info[0].unit_name)
print("5 US survey feet =", 5 * 1200 / 3937, "m  (5 international feet = 1.524 m exactly)")

**5. Two PROJ versions.** Take the first vertex of a dam in the extract (the dams are MultiPolygons: `geometry.iloc[0].geoms[0].exterior.coords[0]`). Transform it to EPSG:4326 with `pyproj` (PROJ 9.5) and with `rasterio.warp.transform` (PROJ 9.8), first using the CRS written in the file, then using `+proj=eqc +lat_ts=-52.28859923 +datum=WGS84 +units=m`. Which pairs agree and which are 30 km apart? Why does the file in the extract not show the problem?

> **From the chapter.** The Case, "the ARC System zone, 30 km apart": the same description is read on a sphere by older PROJ and on an ellipsoid by newer PROJ. `pyproj` and `rasterio` each carry their own PROJ ([chapter 0](../../00_setup.md), §3).
>
> **Code.** No pipeline code: `pyproj` and `rasterio.warp.transform`. In the pipeline, `field._same_place` in [`data_acquisition/sources/field.py`](../../../../data_acquisition/sources/field.py) makes this comparison.

In [ ]:
import pyproj
path = EXTRACT / "2018-07-26/annotations/dams.gpkg"
dams = pyogrio.read_dataframe(path)
x, y = dams.geometry.iloc[0].geoms[0].exterior.coords[0][:2]
print("PROJ in pyproj:", pyproj.proj_version_str, "| PROJ in rasterio:", rasterio.__proj_version__)
in_file = CRS.from_user_input(pyogrio.read_info(path)["crs"]).sub_crs_list[0].to_wkt()
ellipsoid = "+proj=eqc +lat_ts=-52.28859923 +datum=WGS84 +units=m"
for name, crs in (("CRS in the file", in_file), ("ellipsoid string", ellipsoid)):
    a = Transformer.from_crs(CRS.from_user_input(crs), "EPSG:4326", always_xy=True).transform(x, y)
    b = transform(rasterio.crs.CRS.from_user_input(crs), "EPSG:4326", [x], [y])
    print(f"{name:17s} pyproj {a[0]:.4f}, {a[1]:.4f}   rasterio {b[0][0]:.4f}, {b[1][0]:.4f}")
# The file's CRS spells the sphere out (+R=6378137), so both PROJ versions agree. The plain ellipsoid string
# is read with the spherical formula by PROJ 9.5 and with the ellipsoidal one by PROJ 9.8: 30 km apart.

**6. Find it in the code.** Which function writes the WKT2 of a raster into its STAC item when there is no EPSG code? Which flag does `curate` raise for a raster whose CRS differs from the site's, and in which function? Use `inspect` or search the files.

> **From the chapter.** §3 for the WKT2 in an item, and §2 for the flags on a CRS that differs from the site's.
>
> **Code.** Look in [`data_acquisition/catalog/stac.py`](../../../../data_acquisition/catalog/stac.py) and [`data_acquisition/steps/curate.py`](../../../../data_acquisition/steps/curate.py) (the starter cell imports them as `stac` and `curate`).

In [ ]:
import inspect
from data_acquisition.catalog import stac
from data_acquisition.steps import curate
for module, needle in ((stac, "proj.wkt2"), (curate, '"crs_mismatch"')):
    for name, fn in inspect.getmembers(module, inspect.isfunction):
        if fn.__module__ == module.__name__ and needle in inspect.getsource(fn):
            print(f"{module.__name__}.{name} contains {needle}")
# stac.raster_item (and stac.vector_item, for vectors) writes proj.wkt2; curate.check_raster raises crs_mismatch.